# JVC Real Estate Scraper 

Scope:
- Scrape ~1 page of apartment rental listings in JVC (Dubai) from Bayut.com using Selenium.

Why Selenium + undetected-chromedriver?
- Bayut actively blocks scraping through:
    - Captchas
    - Cloudflare bot detection
        - Script-generated modals / onboarding tours
    - Normal requests / standard chromedriver fail.
    - undetected-chromedriver emulates real human browsing.

Why running locally?
- UI debugging is easier (you see modals + overlays).
- You avoid cloud-based automated browser detection.

Output:
- Clean pandas dataframe
- CSV file for use in Google Colab later

Notes:
- We scrape the "Popular" listings by default (URL is the main JVC rental page). 
If you want "Most Recent", you would append "?sort=recent" to the base URL.
- Only essential columns are kept: title, price, frequency, bedrooms, bathrooms, area, location, url.
- Columns like images, agent info, logos are omitted because they are inconsistent or empty.

### Imports

In [ ]:
import time
import os
import random
import pandas as pd
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

`selenium` + `undetected-chromedriver` are used to launch a real browser (Chrome) in order to load the page as a “real user”

### Helper Functions

In [2]:
def build_page_url(base_url, page_num):
    """
    Builds the URL for a specific page.
    If page_num = 1 → return base_url
    Else -> append "?page=page_num"
    
    Used for Pagination
    """
    return base_url if page_num == 1 else f"{base_url}?page={page_num}"

In [3]:
# Helper to extract text via aria-label
def extract(soup, aria):
    el = soup.find(attrs={"aria-label": aria})
    return el.get_text(strip=True) if el else None

_these functions handle popups/modals and overlay tips_

In [4]:
def click_safe(driver, element):
    """
    Tries normal click first, falls back to JS click if needed.
    Used to close modals or popups.
    """
    try:
        element.click()
    except:
        driver.execute_script("arguments[0].click();", element)


In [5]:
def close_all_modals(driver):
    """
    Handles popup modals that Bayut sometimes shows.
    We look for:
        - modal containers
        - close buttons / 'X' buttons
    """
    try:
        modals = driver.find_elements(
            By.XPATH,
            '//div[contains(@class, "_49f04cdc") or contains(@class,"onboarding") or contains(@class, "_07c05f81")]'
        )
        for modal in modals:
            try:
                # close_btns = modal.find_elements(By.XPATH, './/*[contains(@aria-label,"Close") or contains(@class,"37d9afbd")]')
                close_btns = modal.find_elements(By.XPATH, '//*[contains(@aria-label,"close") or contains(@aria-label,"Close") or contains(@class,"close")]')
                for btn in close_btns:
                    if btn.is_displayed():
                        click_safe(driver, btn)
                        print("Closed modal.")
                        time.sleep(1)
            except StaleElementReferenceException:
                pass
    except:
        pass

In [6]:
def dismiss_overlay_by_clicking_body(driver):
    """
    Some Bayut overlays disappear just by clicking anywhere.
    We simulate a user click via JS.
    """
    try:
        driver.execute_script("document.body.click();")
        print("Clicked body to dismiss overlays.")
        time.sleep(1)
    except:
        pass

In [7]:
def handle_overlays(driver):
    """Consolidates all overlay/modal handling into one call."""
    close_all_modals(driver)
    dismiss_overlay_by_clicking_body(driver)

    # simulate a small scroll to trigger anti-bot overlay removal
    try:
        driver.execute_script("window.scrollBy(0, 200);")
        print("Scrolled to remove overlays.")
        time.sleep(1)
    except:
        pass

### Launch Selenium

In [ ]:
# object configuration
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
driver = uc.Chrome(options=options) # initialize the navigator


# Base URL for JVC apartments
# Popular listings (default)
base_url = "https://www.bayut.com/to-rent/property/dubai/jumeirah-village-circle-jvc/"
# If you want Most Recent, use: url = url + "?sort=recent"


NUM_PAGES = 100  # adjust as needed
all_data = []   # all the data we will scrape and parse

wait = WebDriverWait(driver, 20)    # wait for elements to fully load

### Extract Listing Blocks

_Before scraping, we define the data folder_

In [ ]:
# Define data folder relative to notebooks folder
data_folder = os.path.join("..", "data")
os.makedirs(data_folder, exist_ok=True)  # make sure folder exists


In [ ]:
for page in range(1, NUM_PAGES + 1):
    page_url = build_page_url(base_url, page)
    driver.get(page_url)

    # Wait for at least ONE listing to load (much safer and reliable than sleep)
    wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "li[role='article']"))
    )

    handle_overlays(driver)

    wait.until(
        # now we wait to get all the list tags wrapped article tags, 
        # because all the listings are wrapped by the article tag.
        # so we wait for all of the to load before proceeding
        EC.presence_of_all_elements_located((By.XPATH, '//li[@role="article"]'))
    )

    cards = driver.find_elements(By.XPATH, '//li[@role="article"]') # then we get them
    print(f"Page {page} → found {len(cards)} listings") # feedback

    # loop and extract the info
    for idx, card in enumerate(cards, start=1):
        try:

            soup = BeautifulSoup(card.get_attribute("innerHTML"), "html.parser")


            title = extract(soup, "Title")
            price = extract(soup, "Price")
            frequency = extract(soup, "Frequency")
            bedrooms = extract(soup, "Beds")
            bathrooms = extract(soup, "Baths")

            area_tag = soup.find("h4", class_="_60820635")
            area = area_tag.get_text(strip=True) if area_tag else None

            loc_tag = soup.find("h3")
            location = loc_tag.get_text(strip=True) if loc_tag else None


            # URL extraction with debugging
            url_listing = None

            # First try aria-label version (preferred)
            a_tag = soup.find("a", attrs={"aria-label": "Listing link", "href": True})

            # Fallback: any <a> with an href
            if not a_tag:
                a_tag = soup.find("a", href=True)

            if a_tag:
                href = a_tag.get("href")

                # Keep only real property listing URLs
                if href and ("/property" in href or "/listing" in href):
                    if href.startswith("/"):
                        url_listing = "https://www.bayut.com" + href
                    else:
                        url_listing = href


            all_data.append({
                "title": title,
                "price": price,
                "frequency": frequency,
                "bedrooms": bedrooms,
                "bathrooms": bathrooms,
                "area": area,
                "location": location,
                "url": url_listing
            })

        except Exception as e:
            print(f"Error parsing card {idx} on page {page}: {e}")


    print(f"After page {page} -> total rows: {len(all_data)}")


    # backup 
    BACKUP_FREQUENCY = 5  # after how many scraped pages we backup
    if page % BACKUP_FREQUENCY == 0 and page != NUM_PAGES:

        # Backup CSV file
        backup_file_path = os.path.join(data_folder, f"backup_page_{page}.csv")
        pd.DataFrame(all_data).to_csv(backup_file_path, index=False)

    time.sleep(random.uniform(3, 7))

driver.quit()

Clicked body to dismiss overlays.
Scrolled to remove overlays.
Page 1 → found 24 listings
After page 1 -> total rows: 24
Clicked body to dismiss overlays.
Scrolled to remove overlays.
Page 2 → found 24 listings
After page 2 -> total rows: 48
Clicked body to dismiss overlays.
Scrolled to remove overlays.
Page 3 → found 24 listings
After page 3 -> total rows: 72
Clicked body to dismiss overlays.
Scrolled to remove overlays.
Page 4 → found 24 listings
After page 4 -> total rows: 96
Clicked body to dismiss overlays.
Scrolled to remove overlays.
Page 5 → found 24 listings
After page 5 -> total rows: 120


### Create DataFrame + Save

In [ ]:
# Convert scraped data into DataFrame
df = pd.DataFrame(all_data)


# Save DataFrame to CSV
file_path = os.path.join(data_folder, "jvc_apartments.csv")
df.to_csv(file_path, index=False)
print(f"Data saved to {file_path}")

,title,price,frequency,bedrooms,bathrooms,area,location,url
0,Stylish 1BR Fully Furnished I Smart Home I Rea...,"79,999",Yearly,1,2,638 sqft,"Binghatti Heights, JVC District 10, Jumeirah V...",https://www.bayut.com/property/details-1354556...
1,Upgraded Unit| Brand New 1BR With Pool View l ...,"90,000",Yearly,1,2,793 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1319355...
2,Unique Layout I Brand New Building I Closed Ki...,"119,990",Yearly,2,3,"1,300 sqft","SH Living 1, JVC District 14, Jumeirah Village...",https://www.bayut.com/property/details-1296493...
3,Spacious Size | Kitchen Appliances | Vacant,"65,000",Yearly,1,2,831 sqft,"Roxana Residence D, Roxana Residences, JVC Dis...",https://www.bayut.com/property/details-1359705...
4,Unfurnished | Bright and Spacious | Well Maint...,"75,000",Yearly,1,2,767 sqft,"Binghatti Jasmine, JVC District 15, Jumeirah V...",https://www.bayut.com/property/details-1321519...
...,...,...,...,...,...,...,...,...
115,Fully Furnished | Ready to Move | Bills Included,"65,000",Yearly,None,1,482 sqft,"Maple 1, Emirates Gardens 2, JVC District 14, ...",https://www.bayut.com/property/details-1354740...
116,Pool-Park View | Corner Unit | High Floor,"65,000",Yearly,None,1,428 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1339691...
117,Fully Furnished I Spacious Unit | Community View,"70,000",Yearly,1,2,682 sqft,"La Riviera Estate A, La Riviera Estates, JVC D...",https://www.bayut.com/property/details-1351913...
118,Bright & Spacious unit I High floor I Pool vie...,"74,999",Yearly,1,2,798 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1342105...
